# PD AI 도구 — Colab GPU Stable Diffusion 서버

`hooking-point.pages.dev`의 **AI TOOLS > 컨셉 이미지** 탭이 여기서 돌아가는 Stable Diffusion을 호출합니다.

**사용 순서**
1. 상단 메뉴 `런타임 > 런타임 유형 변경`에서 **GPU**(T4)를 선택합니다.
2. [ngrok.com](https://ngrok.com)에 무료 가입 후, 대시보드에서 **Authtoken**을 복사합니다.
3. 아래 셀을 위에서부터 순서대로 실행합니다. 두 번째 코드 셀(SHARED_SECRET)과 마지막 셀의 출력(ngrok URL)을 복사해 Cloudflare Pages 프로젝트의 환경변수로 등록합니다: `COLAB_SHARED_SECRET`, `COLAB_ENDPOINT_URL`.
4. **이 탭을 닫거나 런타임이 끊기면 이미지 생성이 멈추고 자동으로 목업 이미지로 대체됩니다.** 다시 켤 때마다 ngrok URL이 바뀌므로 `COLAB_ENDPOINT_URL`을 다시 업데이트해야 합니다.
5. Colab 무료 플랜은 GPU 세션이 일정 시간(보통 몇 시간) 후 자동 종료됩니다 — 필요할 때마다 다시 실행하세요.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors flask flask-cors pyngrok

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

# 가볍고 빠른 기본 모델입니다. 품질을 더 높이려면 "stabilityai/stable-diffusion-2-1"로 바꿔보세요(로딩이 더 오래 걸립니다).
MODEL_ID = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)
print("모델 로드 완료:", MODEL_ID)

In [ ]:
import secrets

# 아무나 이 서버를 호출하지 못하도록 막는 공유 비밀값입니다.
# 출력된 값을 Cloudflare Pages 환경변수 COLAB_SHARED_SECRET에 그대로 붙여넣으세요.
SHARED_SECRET = secrets.token_urlsafe(24)
print("COLAB_SHARED_SECRET =", SHARED_SECRET)

In [ ]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "여기에 ngrok.com 대시보드에서 발급받은 Authtoken을 붙여넣으세요"  # @param {type:"string"}
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
import base64
import io
import threading

from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)


@app.route("/generate", methods=["POST"])
def generate():
    if SHARED_SECRET and request.headers.get("X-Shared-Secret") != SHARED_SECRET:
        return jsonify({"error": "unauthorized"}), 401

    data = request.get_json(force=True) or {}
    prompt = (data.get("prompt") or "").strip()
    if not prompt:
        return jsonify({"error": "prompt가 비어 있습니다."}), 400

    image = pipe(prompt, num_inference_steps=28, guidance_scale=7.5).images[0]
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    image_base64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return jsonify({"image_base64": image_base64})


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})


public_url = ngrok.connect(5000)
print("COLAB_ENDPOINT_URL =", public_url)

threading.Thread(target=lambda: app.run(port=5000)).start()

위 셀을 실행한 뒤 출력된 `COLAB_ENDPOINT_URL`과 `COLAB_SHARED_SECRET`을 Cloudflare Pages 대시보드 > 프로젝트 선택 > Settings > Environment variables에 등록하면, 사이트의 AI TOOLS > 컨셉 이미지 탭이 이 Colab GPU를 사용해 실제 이미지를 생성합니다.

이 노트북 탭을 켜둔 동안에만 동작하며, 런타임이 끊기면 자동으로 목업 이미지로 대체됩니다(에러 없이 안전하게 대체됨).